# Single-, Two-, and Three-Layer Neural Networks for Regression Using PyTorch Built-In Activations

This notebook mirrors the from-scratch neural network notebook, but uses PyTorch to define and train the models.

The main difference from a manual implementation is that PyTorch handles:

- the affine layers with `torch.nn.Linear`,
- the activation functions with built-in PyTorch modules such as `torch.nn.Tanh`, `torch.nn.ReLU`, `torch.nn.Sigmoid`, and `torch.nn.Identity`,
- the loss function with `torch.nn.MSELoss`,
- backpropagation with `loss.backward()`, and
- gradient descent with `torch.optim.SGD`.

We will compare:

1. a single-layer neural network,
2. a two-layer neural network, and
3. a three-layer neural network.

Here, the number of layers means the number of trainable affine transformations, not the number of activation functions.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# For reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 1. Generate the Regression Data

We will use the same simple nonlinear regression example as before:

$$
f(x) = 3\sin(1.5x) + x^2.
$$

The training set contains only a small number of randomly sampled observations, so the models have to learn from limited data.

In [ ]:
# Number of observations
m = 10

# Regression function
f = lambda x: 3 * np.sin(1.5 * x) + x**2

# Generate training data
X = np.random.uniform(-5, 5, (1, m))
Y = f(X)

# Generate testing grid
X_test = np.linspace(-6, 6, 100).reshape(1, -1)
Y_test = f(X_test)

# Make sure Y has shape [1, m]
Y = Y.reshape(1, -1)

# Convert NumPy arrays to PyTorch tensors.
# PyTorch's nn.Linear expects shape [number of samples, number of features],
# so we transpose from [features, samples] to [samples, features].
X_tensor = torch.tensor(X.T, dtype=torch.float32)
Y_tensor = torch.tensor(Y.T, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.T, dtype=torch.float32)

print('X_tensor shape:', X_tensor.shape)
print('Y_tensor shape:', Y_tensor.shape)
print('X_test_tensor shape:', X_test_tensor.shape)

In [ ]:
plt.figure()
plt.plot(X.flatten(), Y.flatten(), 'ro', markerfacecolor='r')
plt.plot(X_test.flatten(), Y_test.flatten(), '--k', linewidth=2)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Regression Dataset')
plt.grid(True)
plt.legend(['training data', 'true signal'])
plt.show()

## 2. Training Options

These settings are chosen to resemble the original from-scratch implementation.

The hidden-layer activation functions will be created directly using PyTorch's built-in activation modules.

In [ ]:
input_size = 1
output_size = 1
hidden_nodes = 10
hidden_nodes_2 = 10

learning_rate = 1e-3
epochs = 4000
batch_size = 2
verbose = True

# Built-in PyTorch activation modules.
# Try replacing nn.Tanh() with nn.ReLU() or nn.Sigmoid().
hidden_activation = nn.Tanh()
output_activation = nn.Identity()   # Identity means linear output activation.

## 3. Single-Layer Neural Network

A single-layer neural network for regression has one trainable affine transformation:

$$
\hat{y} = \sigma_{out}(Wx + b).
$$

For standard regression, the output activation is usually linear. In PyTorch, we can represent a linear activation using `nn.Identity()`.

In [ ]:
class SingleLayerNN(nn.Module):
    def __init__(self, input_size, output_size, output_activation=None):
        super().__init__()
        
        if output_activation is None:
            output_activation = nn.Identity()
        
        self.linear = nn.Linear(input_size, output_size)
        self.output_activation = output_activation

    def forward(self, x):
        z = self.linear(x)
        yhat = self.output_activation(z)
        return yhat

## 4. Two-Layer Neural Network

A two-layer neural network has one hidden layer and one output layer:

$$
A^{[1]} = \sigma(W^{[1]}X + b^{[1]}),
$$

$$
\hat{Y} = W^{[2]}A^{[1]} + b^{[2]}.
$$

The code below uses PyTorch built-in activation modules directly. For example, the hidden activation can be `nn.Tanh()`, `nn.ReLU()`, or `nn.Sigmoid()`.

In [ ]:
class TwoLayerNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size,
                 hidden_activation=None, output_activation=None):
        super().__init__()
        
        if hidden_activation is None:
            hidden_activation = nn.Tanh()
        if output_activation is None:
            output_activation = nn.Identity()
        
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.hidden_activation = hidden_activation
        self.layer2 = nn.Linear(hidden_size, output_size)
        self.output_activation = output_activation

    def forward(self, x):
        z1 = self.layer1(x)
        a1 = self.hidden_activation(z1)
        z2 = self.layer2(a1)
        yhat = self.output_activation(z2)
        return yhat

## 5. Three-Layer Neural Network

Here, a three-layer neural network means the model has three trainable affine layers:

1. input to first hidden layer,
2. first hidden layer to second hidden layer,
3. second hidden layer to output.

The model is:

$$
A^{[1]} = \sigma_1(W^{[1]}X + b^{[1]}),
$$

$$
A^{[2]} = \sigma_2(W^{[2]}A^{[1]} + b^{[2]}),
$$

$$
\hat{Y} = W^{[3]}A^{[2]} + b^{[3]}.
$$

Again, the hidden activations are PyTorch modules such as `nn.Tanh()` or `nn.ReLU()`.

In [ ]:
class ThreeLayerNN(nn.Module):
    def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size,
                 hidden_activation_1=None, hidden_activation_2=None, output_activation=None):
        super().__init__()
        
        if hidden_activation_1 is None:
            hidden_activation_1 = nn.Tanh()
        if hidden_activation_2 is None:
            hidden_activation_2 = nn.Tanh()
        if output_activation is None:
            output_activation = nn.Identity()
        
        self.layer1 = nn.Linear(input_size, hidden_size_1)
        self.hidden_activation_1 = hidden_activation_1
        self.layer2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.hidden_activation_2 = hidden_activation_2
        self.layer3 = nn.Linear(hidden_size_2, output_size)
        self.output_activation = output_activation

    def forward(self, x):
        z1 = self.layer1(x)
        a1 = self.hidden_activation_1(z1)
        z2 = self.layer2(a1)
        a2 = self.hidden_activation_2(z2)
        z3 = self.layer3(a2)
        yhat = self.output_activation(z3)
        return yhat

## 6. Optional: The Same Models with `nn.Sequential`

PyTorch also lets us define the networks using `nn.Sequential`, where layers are listed in the order they are applied.

This is often the cleanest way to build simple feedforward networks.

In [ ]:
# These are equivalent-style definitions using nn.Sequential.

single_model_sequential = nn.Sequential(
    nn.Linear(input_size, output_size),
    nn.Identity()
)

two_model_sequential = nn.Sequential(
    nn.Linear(input_size, hidden_nodes),
    nn.Tanh(),
    nn.Linear(hidden_nodes, output_size),
    nn.Identity()
)

three_model_sequential = nn.Sequential(
    nn.Linear(input_size, hidden_nodes),
    nn.Tanh(),
    nn.Linear(hidden_nodes, hidden_nodes_2),
    nn.Tanh(),
    nn.Linear(hidden_nodes_2, output_size),
    nn.Identity()
)

print(single_model_sequential)
print(two_model_sequential)
print(three_model_sequential)

## 7. Training Function

This training loop works for any PyTorch model.

For each mini-batch, it:

1. computes predictions,
2. evaluates the MSE loss,
3. clears old gradients,
4. computes new gradients using `loss.backward()`, and
5. updates parameters using stochastic gradient descent.

In [ ]:
def train_pytorch_nn(model, X_train, Y_train, learning_rate=1e-3, epochs=4000,
                     batch_size=2, verbose=True):
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    
    n_samples = X_train.shape[0]
    history = {'loss': np.zeros(epochs)}
    
    for epoch in range(epochs):
        permutation = torch.randperm(n_samples)
        epoch_loss = 0.0
        num_batches = int(np.ceil(n_samples / batch_size))
        
        for b in range(num_batches):
            idx_start = b * batch_size
            idx_end = min((b + 1) * batch_size, n_samples)
            batch_indices = permutation[idx_start:idx_end]
            
            Xb = X_train[batch_indices]
            Yb = Y_train[batch_indices]
            
            # Forward pass
            Yhat = model(Xb)
            loss = criterion(Yhat, Yb)
            
            # Backward pass and update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        history['loss'][epoch] = epoch_loss / num_batches
        
        if verbose and (((epoch + 1) % max(1, epochs // 10) == 0) or (epoch == 0)):
            print(f'Epoch {epoch + 1}/{epochs}, Loss = {history["loss"][epoch]:.6f}')
    
    return history

## 8. Train the Single-Layer Network

This model has one trainable layer. With `nn.Identity()` as the output activation, it is equivalent to linear regression.

In [ ]:
torch.manual_seed(1)

single_model = SingleLayerNN(
    input_size=input_size,
    output_size=output_size,
    output_activation=nn.Identity()
)

single_history = train_pytorch_nn(
    single_model,
    X_tensor,
    Y_tensor,
    learning_rate=learning_rate,
    epochs=epochs,
    batch_size=batch_size,
    verbose=verbose
)

with torch.no_grad():
    single_yhat = single_model(X_test_tensor).numpy().T

In [ ]:
plt.figure()
plt.plot(single_history['loss'], linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Single-Layer Network Training Loss')
plt.grid(True)
plt.show()

plt.figure()
plt.plot(X.flatten(), Y.flatten(), 'ro', markerfacecolor='r')
plt.plot(X_test.flatten(), Y_test.flatten(), '--k', linewidth=2)
plt.plot(X_test.flatten(), single_yhat.flatten(), ':b', linewidth=2)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Single-Layer PyTorch Network Prediction')
plt.grid(True)
plt.legend(['data', 'true signal', 'single-layer prediction'])
plt.show()

## 9. Train the Two-Layer Network

This model has one hidden layer with a built-in PyTorch activation function.

Below, we use `nn.Tanh()` as the hidden activation. You can change this to `nn.ReLU()` or `nn.Sigmoid()`.

In [ ]:
torch.manual_seed(1)

two_model = TwoLayerNN(
    input_size=input_size,
    hidden_size=hidden_nodes,
    output_size=output_size,
    hidden_activation=nn.Tanh(),
    output_activation=nn.Identity()
)

two_history = train_pytorch_nn(
    two_model,
    X_tensor,
    Y_tensor,
    learning_rate=learning_rate,
    epochs=epochs,
    batch_size=batch_size,
    verbose=verbose
)

with torch.no_grad():
    two_yhat = two_model(X_test_tensor).numpy().T

In [ ]:
plt.figure()
plt.plot(two_history['loss'], linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Two-Layer Network Training Loss')
plt.grid(True)
plt.show()

plt.figure()
plt.plot(X.flatten(), Y.flatten(), 'ro', markerfacecolor='r')
plt.plot(X_test.flatten(), Y_test.flatten(), '--k', linewidth=2)
plt.plot(X_test.flatten(), two_yhat.flatten(), ':b', linewidth=2)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Two-Layer PyTorch Network Prediction')
plt.grid(True)
plt.legend(['data', 'true signal', 'two-layer prediction'])
plt.show()

## 10. Train the Three-Layer Network

This model has two hidden layers and one output layer. Both hidden layers use PyTorch built-in activation functions.

In [ ]:
torch.manual_seed(1)

three_model = ThreeLayerNN(
    input_size=input_size,
    hidden_size_1=hidden_nodes,
    hidden_size_2=hidden_nodes_2,
    output_size=output_size,
    hidden_activation_1=nn.Tanh(),
    hidden_activation_2=nn.Tanh(),
    output_activation=nn.Identity()
)

three_history = train_pytorch_nn(
    three_model,
    X_tensor,
    Y_tensor,
    learning_rate=learning_rate,
    epochs=epochs,
    batch_size=batch_size,
    verbose=verbose
)

with torch.no_grad():
    three_yhat = three_model(X_test_tensor).numpy().T

In [ ]:
plt.figure()
plt.plot(three_history['loss'], linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Three-Layer Network Training Loss')
plt.grid(True)
plt.show()

plt.figure()
plt.plot(X.flatten(), Y.flatten(), 'ro', markerfacecolor='r')
plt.plot(X_test.flatten(), Y_test.flatten(), '--k', linewidth=2)
plt.plot(X_test.flatten(), three_yhat.flatten(), ':b', linewidth=2)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Three-Layer PyTorch Network Prediction')
plt.grid(True)
plt.legend(['data', 'true signal', 'three-layer prediction'])
plt.show()

## 11. Compare All Three Models

The single-layer model is linear, while the two-layer and three-layer models can learn nonlinear behavior because they include nonlinear hidden activations.

The activation functions used here are built-in PyTorch modules:

- `nn.Tanh()` for the hidden layers,
- `nn.Identity()` for the linear output layer.

In [ ]:
plt.figure()
plt.plot(single_history['loss'], linewidth=1.5)
plt.plot(two_history['loss'], linewidth=1.5)
plt.plot(three_history['loss'], linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.grid(True)
plt.legend(['single-layer', 'two-layer', 'three-layer'])
plt.show()

plt.figure()
plt.plot(X.flatten(), Y.flatten(), 'ro', markerfacecolor='r')
plt.plot(X_test.flatten(), Y_test.flatten(), '--k', linewidth=2)
plt.plot(X_test.flatten(), single_yhat.flatten(), linewidth=2)
plt.plot(X_test.flatten(), two_yhat.flatten(), linewidth=2)
plt.plot(X_test.flatten(), three_yhat.flatten(), linewidth=2)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Single-Layer vs Two-Layer vs Three-Layer PyTorch Regression')
plt.grid(True)
plt.legend(['data', 'true signal', 'single-layer', 'two-layer', 'three-layer'])
plt.show()

print('Final single-layer loss:', single_history['loss'][-1])
print('Final two-layer loss:', two_history['loss'][-1])
print('Final three-layer loss:', three_history['loss'][-1])

## 12. Trying Other Built-In Activations

Because the models accept PyTorch activation modules directly, changing activations is simple.

For example, a two-layer ReLU network can be created with:

```python
two_relu_model = TwoLayerNN(
    input_size=input_size,
    hidden_size=hidden_nodes,
    output_size=output_size,
    hidden_activation=nn.ReLU(),
    output_activation=nn.Identity()
)
```

A three-layer network with different activations in each hidden layer can be created with:

```python
three_mixed_model = ThreeLayerNN(
    input_size=input_size,
    hidden_size_1=hidden_nodes,
    hidden_size_2=hidden_nodes_2,
    output_size=output_size,
    hidden_activation_1=nn.Tanh(),
    hidden_activation_2=nn.ReLU(),
    output_activation=nn.Identity()
)
```

## 13. Summary

In this notebook, we used PyTorch's built-in neural network tools rather than manually coding backpropagation.

The key tools were:

- `nn.Linear` for affine transformations,
- `nn.Tanh`, `nn.ReLU`, `nn.Sigmoid`, and `nn.Identity` for activation functions,
- `nn.MSELoss` for the regression loss,
- `torch.optim.SGD` for gradient descent, and
- `loss.backward()` for automatic differentiation.

This is the standard PyTorch workflow for building and training feedforward neural networks.